In [ ]:
!pip install monai

In [ ]:
import pandas as pd
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch.amp import GradScaler, autocast # 속도 및 메모리 최적화
from monai.transforms import (
    Compose, MapTransform, SelectItemsd, RandFlipd, RandAffined,
    RandGaussianNoised, RandAdjustContrastd,
    RandGaussianSmoothd, Transposed, ToTensord
)
from monai.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import torchmetrics # AUC 계산을 쉽게 해주는 라이브러리
from torch.optim.lr_scheduler import ReduceLROnPlateau
import pickle  #파일저장에
from tqdm import tqdm  # 학습 진행 상황 시각화를 위해 추가
from torch.optim import AdamW
import timm
from collections import OrderedDict

# 0. 설정 및 경로
BASE_DIR = '/kaggle/input/rsna-2023-abdominal-trauma-detection/'
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_EPOCHS = 25
CLASS_NAME_LIST = ['bowel', 'extravasation', 'kidney', 'liver', 'spleen', 'any_injury']
LEARNING_RATE = 1e-4
LAYER_DECAY = 0.8

MONAI_MODEL_SAVE_PATH = '/kaggle/working/monai_ct_convnext_v9.pth'

MONAI_MODEL_SAVE_PATH_CONTINUE = ''


IMAGE_TARGET = (64,224,224)
NUM_SLICES = 64

# 전처리된 데이터를 저장할 폴더
SAVE_DIR = '/kaggle/input/rsna-2023-atd-preprocessed-s224/result/'
os.makedirs(SAVE_DIR, exist_ok=True)

# 파일 읽기
train_df = pd.read_csv(f'{BASE_DIR}train_2024.csv') # 파일명 확인 필요 (보통 train.csv)
tags_df = pd.read_parquet(f'{BASE_DIR}train_dicom_tags.parquet')

# 고유 폴더 경로 추출 및 환자 ID 연결
tags_df['series_path'] = tags_df['path'].str.split('/').str[:-1].str.join('/')
unique_series = tags_df[['PatientID', 'series_path']].drop_duplicates()

data_dicts = []
for idx, row in unique_series.iterrows():
    p_id = int(row['PatientID'])
    s_path = row['series_path']

    # 해당 환자의 라벨 정보 가져오기
    patient_labels = train_df[train_df['patient_id'] == p_id]
    if len(patient_labels) == 0: continue # 라벨 없는 경우 제외
    labels = patient_labels.iloc[0]

    data_dicts.append({
        "image": f"{BASE_DIR}{s_path}",
        "patient_id": p_id,

        # 2진 분류 (Healthy, Injury) -> [1, 0] 또는 [0, 1] 형태가 됨
        "bowel": labels[['bowel_healthy', 'bowel_injury']].values.astype("float32"),
        "extravasation": labels[['extravasation_healthy', 'extravasation_injury']].values.astype("float32"),

        # 3중 분류 (Healthy, Low, High) -> [1, 0, 0], [0, 1, 0], [0, 0, 1] 형태가 됨
        "liver": labels[['liver_healthy', 'liver_low', 'liver_high']].values.astype("float32"),
        "kidney": labels[['kidney_healthy', 'kidney_low', 'kidney_high']].values.astype("float32"),
        "spleen": labels[['spleen_healthy', 'spleen_low', 'spleen_high']].values.astype("float32"),

        # any_injury가 1이면 "어딘가 이상함", 0이면 "완전 건강"
        "any_injury": np.array([1 - labels['any_injury'], labels['any_injury']]).astype("float32")

    })

patient_ids = train_df['patient_id'].unique()
train_ids, val_ids = train_test_split(patient_ids, test_size=0.2, random_state=42)
train_files = [d for d in data_dicts if d['patient_id'] in train_ids] # data_dicts에 patient_id 키 추가 필요
val_files = [d for d in data_dicts if d['patient_id'] in val_ids]

print(f"준비된 데이터 수: {len(data_dicts)}")
print(f"디바이스: {DEVICE}")

In [ ]:
# ==========================================
# 1. LLRD 옵티마이저 설정 (6개 그룹)
# ==========================================
def get_optimizer_with_llrd(model, base_lr=1e-4, weight_decay=0.05, layer_decay=0.8):
    raw_model = model.module if hasattr(model, 'module') else model
    param_groups = []
    
    # 1. Head 그룹
    head_modules = [raw_model.transformer_encoder, raw_model.attention_net, 
                    raw_model.suspicion_head, raw_model.organ_heads, raw_model.gated_norm]
    head_params = []
    for m in head_modules:
        head_params.extend([p for p in m.parameters() if p.requires_grad])
    if raw_model.position_embedding.requires_grad:
        head_params.append(raw_model.position_embedding)

    param_groups.append({"params": head_params, "lr": base_lr, "weight_decay": weight_decay, "name": "head"})

    # 2. Backbone Stages (역순: Stage3 -> Stem)
    backbone = raw_model.backbone
    stages = [backbone.stem, backbone.stages[0], backbone.stages[1], backbone.stages[2], backbone.stages[3]]
    stages.reverse() 

    for i, stage in enumerate(stages):
        ratio = layer_decay ** (i + 1)
        stage_params = [p for p in stage.parameters() if p.requires_grad]
        if len(stage_params) > 0:
            param_groups.append({"params": stage_params, "lr": base_lr * ratio, "weight_decay": weight_decay, "name": f"backbone_layer_{4-i}"})

    return AdamW(param_groups)

class Timm_Model(torch.nn.Module):
    def __init__(self, model_name='convnext_tiny', num_slices=64):
        super().__init__()
        # 특징 추출기 (ConvNeXt)
        # num_classes = 1000 (기본값): 모델의 최종 출력이 1,000개의 숫자(카테고리 점수)로 나옵니다.
        # num_classes = 0: 1,000개를 맞히는 마지막 층을 아예 없애버립니다. 대신, 그 바로 직전 단계인 **'이미지의 핵심 특징 정보(Feature Vector)'**를 그대로 출력합니다.
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)


        for param in self.backbone.parameters():
            param.requires_grad = False

        self.dim = self.backbone.num_features # tiny 기준 768
        self.num_slices = num_slices
        self.gated_norm = nn.LayerNorm(self.dim)

        # Position Encoding (슬라이스 번호 매기기)
        self.position_embedding = nn.Parameter(torch.zeros(1, num_slices, self.dim))
        self.position_dropout = nn.Dropout(0.1)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.dim,
            nhead=8,
            dim_feedforward=self.dim * 2,
            dropout=0.1,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)

        # 어텐션 풀링: 64장 중 수상한 놈을 골라내는 '심사위원'
        self.attention_net = nn.Sequential(
            nn.Linear(self.dim, 256),
            nn.Tanh(),
            nn.Dropout(0.1), # 추가
            nn.Linear(256, 1)
        )

        # "종합 이상 징후 탐지" 전용 헤드 (의심 모델 역할)
        self.suspicion_head = nn.Sequential(
            nn.Linear(self.dim, 256),  # 768개를 256개의 핵심 의심 후보로 압축
            nn.LayerNorm(256),  # 학습을 안정적으로 만들어줌
            nn.ReLU(),            # 중요한 의심 신호만 통과시킴
            nn.Dropout(0.1),      # 과적합 방지 (너무 예민해지는 것 방지)
            nn.Linear(256, 2)     # 최종 경보 [정상, 이상]
        )

        # "정밀 병명 분류" 전용 헤드 (분류 모델 역할)
        # 장기별 결과 2 or 3개 도출
        self.organ_heads = nn.ModuleDict({
            'bowel': nn.Linear(self.dim, 2),
            'extravasation': nn.Linear(self.dim, 2),
            'liver': nn.Linear(self.dim, 3),
            'kidney': nn.Linear(self.dim, 3),
            'spleen': nn.Linear(self.dim, 3)
        })

    # [입력 데이터] (Batch, 64 Slices, 3, 128, 128)
    #     ↓
    # ==========================================================
    # 1. [Backbone: ConvNeXt-Tiny] -> "시각 신경 (이미지 스캐너)"
    # - 64장의 슬라이스를 각각 스캔하여 768차원의 특징 추출
    # - (64, 3, 128, 128) -> (64, 768)
    # ==========================================================
    #     ↓
    # 2. [Position Embedding] -> "인덱스 부여 (공간 좌표)"
    # - 각 특징에 "이건 1번(머리), 이건 64번(골반)"이라는 위치 정보 주입
    # ==========================================================
    #     ↓
    # 3. [Transformer Encoder] -> "종합 분석 (슬라이스 간 대화)"
    # - 64장의 특징들이 서로 정보를 교환하며 전후 맥락 파악
    # - "5번 슬라이스의 상처가 10번까지 이어지네? 큰 부상이다!"
    # ==========================================================
    #     ↓
    # 4. [Attention Pooling] -> "심사위원 (결정적 증거 포착)"
    # - 64장 중 가장 수상한(부상이 의심되는) 슬라이스에 높은 점수 부여
    # - 64개의 특징을 단 1개의 '필살기 특징 벡터'로 압축 (1, 768)
    # ==========================================================
    #     ↓
    # ==========================================================
    # 5. [suspicion_head] -> "응급의학과 의사" (전체 부상 유무 판단)
    # - [부상 확률 (injury_prob)] (0.0 ~ 1.0)
    # ==========================================================
    #     ↓
    # ==========================================================
    # [organ_heads] -> "전문의 진단 (최종 판단)"
    # - Bowel, Liver, Kidney, Spleen 등 정밀 진단
    # - 부상 확률을 곱하기 때문에 부상에 결과에 영향을 받음
    # ==========================================================
    def forward(self, x):
        # 2.5D 방식으로 전체 슬라이스 훑기
        # x shape: (Batch, 64, 3, 128, 128)
        b, s, c, h, w = x.shape

        chunk_size = 8 # 한 번에 처리할 슬라이스 개수 (메모리에 따라 조절)
        all_features = []

        for i in range(0, s, chunk_size):
            # x_chunk: (Batch, 16, 3, 128, 128)
            x_chunk = x[:, i : i + chunk_size]

            # 2D 연산을 위해 일시적으로 배치 차원으로 합침
            x_chunk = x_chunk.reshape(-1, c, h, w) # (Batch*16, 3, 128, 128)

            # 백본 통과 (이 순간 메모리 사용량이 chunk_size만큼으로 제한됨)
            feat_chunk = self.backbone(x_chunk) # (Batch*16, 768)

            # 다시 슬라이스 차원 분리 후 리스트에 저장
            feat_chunk = feat_chunk.view(b, -1, self.dim)

            all_features.append(feat_chunk)

        # 모든 특징 합치기
        features = torch.cat(all_features, dim=1) # (Batch, 64, 768)

        # --- [Step 2] Position Encoding: 위치 정보 주입 ---
        # 데이터가 정렬되어 들어와도, 모델이 이를 '좌표'로 인식하게 함
        features = features + self.position_embedding
        features = self.position_dropout(features)

        # --- [Step 3] Transformer Encoder: 슬라이스 간 상호작용 ---
        # 64장의 슬라이스가 서로의 정보를 참조하여 입체적인 특징으로 진화
        features = self.transformer_encoder(features) # (Batch, 64, 768)

        # Attention Pooling으로 '이상 지점' 증폭
        # 각 슬라이스의 수상함 점수 계산
        att_scores = self.attention_net(features) # (B, 64, 1)

         # 점수를 0~1 사이 비중(가중치)으로 변환
        att_weights = F.softmax(att_scores, dim=1) # (B, S, 1)

        # 가중치를 곱해서 하나로 합침 (가장 수상한 슬라이스 정보가 증폭됨)
        combined = torch.sum(features * att_weights, dim=1) # (B, 768)

        # 결과 도출
        # 1. 먼저 "부상 유무"를 판단합니다.
        injury_logits = self.suspicion_head(combined) # (B, 2)
        # 부상일 확률(Probability)을 구합니다.
        injury_prob = torch.softmax(injury_logits, dim=1)[:, 1:2] # (B, 1)

        # 2. [핵심] 부상 확률을 장기별 특징에 곱해줍니다 (Gating)
        # 부상이 아닐 것 같으면(0에 가까우면) 장기별 점수들도 0에 가까워지도록 강제합니다.
        gated_features = self.gated_norm(combined * injury_prob)

        # 3. 정밀 진단은 이 게이트를 통과한 특징으로 수행합니다.
        out = {k: head(gated_features) for k, head in self.organ_heads.items()}
        out['any_injury'] = injury_logits

        return out


def monai_train_pipeline():
    return Compose([
        LoadNpyTransformd(keys=["image"]),

        # 공간적 변형 (Spatial)
        # spatial_axis: 0=S(Slices), 1=H, 2=W
        RandFlipd(keys=["image"], prob=0.5, spatial_axis=1), # 좌우 반전
        RandFlipd(keys=["image"], prob=0.5, spatial_axis=2), # 상하 반전

        RandAffined(
            keys=["image"],
            prob=0.3,
            # (S, H, W) 각 축에 대한 회전/스케일
            rotate_range=(0.1, 0.1, 0.1),
            scale_range=(0.1, 0.1, 0.1),
            translate_range=(15, 15, 15),
            padding_mode="zeros",
            mode="bilinear"
        ),

        # 강도 및 노이즈 (Intensity)
        RandGaussianNoised(keys=["image"], prob=0.2, mean=0.0, std=0.05),
        RandAdjustContrastd(keys=["image"], prob=0.2, gamma=(0.7, 1.3)),
        RandGaussianSmoothd(keys=["image"], prob=0.1, sigma_x=(0.5, 1.0)),

        # 모델 입력을 위해 다시 원래 차원으로 복구 (S, C, H, W)
        # Timm_Model이 (Batch, Slices, C, H, W)를 기대하므로
        Transposed(keys=["image"], indices=(1, 0, 2, 3)),

        ToTensord(keys=["image"] + CLASS_NAME_LIST),

        SelectItemsd(keys=["image"] + CLASS_NAME_LIST)
    ])


def monai_val_pipeline():
    return Compose([
        LoadNpyTransformd(keys=["image"]),
        Transposed(keys=["image"], indices=(1, 0, 2, 3)),
        ToTensord(keys=["image"] + CLASS_NAME_LIST),
        SelectItemsd(keys=["image"] + CLASS_NAME_LIST)
    ])


class LoadNpyTransformd(MapTransform):
    def __call__(self, data):
        d = dict(data)
        file_path = d["image"]
        try:
            img = np.load(file_path)
            if img.shape[-1] == 3: # (64, 128, 128, 3)인 경우
                img = np.transpose(img, (3, 0, 1, 2)) # (3, 64, 128, 128)
            d["image"] = torch.from_numpy(img).float()
        except Exception as e:
            print(f"\n❌ npy 파일 로드 실패: {file_path} | 에러: {e}")
            raise e
        return d


def evaluate(model, loader, epoch, criterion_dict):
    model.eval()
    val_epoch_loss = 0
    auc_metrics = torch.nn.ModuleDict({
        'bowel': torchmetrics.AUROC(task="multiclass", num_classes=2),
        'extravasation': torchmetrics.AUROC(task="multiclass", num_classes=2),
        'liver': torchmetrics.AUROC(task="multiclass", num_classes=3),
        'kidney': torchmetrics.AUROC(task="multiclass", num_classes=3),
        'spleen': torchmetrics.AUROC(task="multiclass", num_classes=3),
        'any_injury': torchmetrics.AUROC(task="multiclass", num_classes=2)
    }).to(DEVICE)

    with torch.no_grad():
        val_loop = tqdm(loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [Validation]", leave=False)
        for batch in val_loop:
            inputs = batch["image"].to(DEVICE)
            outputs = model(inputs)

            # 분류
            loss = 0
            for k in CLASS_NAME_LIST:
                preds = torch.softmax(outputs[k], dim=1)
                if hasattr(preds, "as_tensor"): # MetaTensor인 경우에만 호출
                    preds = preds.as_tensor()

                # 2. 정답값 처리
                target = batch[k].to(DEVICE)
                if hasattr(target, "as_tensor"): # 여기서 메타데이터 제거
                    target = target.as_tensor()

                # 이미 target이 순수 텐서이므로, 이후 연산 결과(target_idx)도 순수 텐서입니다.
                if target.dim() > 1:
                    target_idx = torch.argmax(target, dim=1)
                else:
                    target_idx = target.long()

                # [수정] target_idx.as_tensor() 줄을 삭제하거나 아래와 같이 안전하게 변경
                # target_idx = target_idx.as_tensor() <- 이 줄이 에러의 원인이었습니다. 삭제하세요.

                # 3. 메트릭 업데이트
                auc_metrics[k].update(preds, target_idx)

                # 4. Loss 계산 (outputs[k]도 안전하게 변환)
                out_k = outputs[k].as_tensor() if hasattr(outputs[k], "as_tensor") else outputs[k]
                loss_func = criterion_dict[k]
                loss += loss_func(out_k, target_idx)

            val_epoch_loss += loss.item()
            val_loop.set_postfix(val_loss=loss.item())

    avg_val_loss = val_epoch_loss / len(loader)

    # AUC 계산
    auc_results = {k: auc_metrics[k].compute().item() for k in CLASS_NAME_LIST}
    for k in CLASS_NAME_LIST: auc_metrics[k].reset()

    return auc_results, avg_val_loss


def process_one_item(item):
    new_item = item.copy()

    # 원본 경로에서 시리즈 ID 추출 (예: train_images/123/456 -> 456)

    s_id = new_item['image'].split("/")[-1]

    # 통합 전처리 파일 경로 예: /kaggle/working/final_output/456.npy
    target_path = os.path.join(SAVE_DIR, f"{s_id}.npy")

    if os.path.isfile(target_path):

        new_item['image'] = target_path
        return new_item

    return None


# Stage 1 (Epoch 0-5): "신입 의사 교육"
# 동결: Backbone (이미지 스캐너)
# 학습: Transformer + Heads (판단 로직)
# 이유: 이미 똑똑한 스캐너는 그대로 두고, 의료 데이터의 특징을 조합해 판단하는 뇌(Heads)부터 먼저 가르칩니다.
# Stage 2 (Epoch 6-10): "전문가와 손발 맞추기"
# 동결 해제: Backbone의 마지막 층(Stage 3)
# 이유: 판단 능력이 생긴 뇌에 맞춰, 가장 정교한 정보를 뽑는 마지막 시각 층을 미세 조정합니다.
# Stage 3 (Epoch 11-20): "팀워크 전체 최적화"
# 동결 해제: 전체 파라미터
# 이유: 모든 층이 한 팀이 되어 의료 영상의 아주 미세한 특징까지 잡아내도록 아주 낮은 학습률로 마무리 훈련을 합니다.
def train(train_files_preprocessed, val_files_preprocessed,
          train_pipeline, val_pipeline,
          model_save_path):


    # 비교하기
    # 구분	    	BCEWithLogitsLoss		      		CrossEntropyLoss
    # 풀네임		    Binary Cross Entropy with Logits		(Multiclass) Cross Entropy
    # 주요 목적		이진 분류 (Yes or No)			    	다중 분류 (A, B, C 중 하나)
    # 출력 노드 수	1개 (0~1 사이의 확률)			    	N개 (각 클래스별 점수)
    # 활성 함수		Sigmoid (내장됨)				    	Softmax (내장됨)
    # 타겟 라벨		0.0 또는 1.0 (Float)			     	0, 1, 2... 인덱스 (Long)
    # 특징		    각 타겟이 독립적임 (Multi-label 가능)	타겟 간 경쟁 관계 (합이 1이 됨)

    train_ds = Dataset(data=train_files_preprocessed, transform=train_pipeline)
    val_ds = Dataset(data=val_files_preprocessed, transform=val_pipeline)

    train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=4, pin_memory=(DEVICE.type == 'cuda') )
    val_loader = DataLoader(val_ds, batch_size=2, shuffle=True, num_workers=4, pin_memory=(DEVICE.type == 'cuda') )

    # 2. 모델, 손실함수, 옵티마이저
    model = Timm_Model(model_name='convnext_tiny').to(DEVICE)
    if torch.cuda.device_count() > 1:
        print("2개의 GPU를 사용합니다.")
        model = nn.DataParallel(model) # 모델을 복사하여 양쪽 GPU에 분산
    raw_model = model.module if hasattr(model, 'module') else model
    
    criterion_dict = {
        'bowel': nn.CrossEntropyLoss(weight=torch.tensor([1.0, 10.0]).to(DEVICE), label_smoothing=0.05),
        'extravasation': nn.CrossEntropyLoss(weight=torch.tensor([1.0, 10.0]).to(DEVICE), label_smoothing=0.05),
        'any_injury': nn.CrossEntropyLoss(weight=torch.tensor([1.0, 10.0]).to(DEVICE), label_smoothing=0.05),
        'liver': nn.CrossEntropyLoss(weight=torch.tensor([1.0, 3.0, 5.0]).to(DEVICE), label_smoothing=0.05),
        'kidney': nn.CrossEntropyLoss(weight=torch.tensor([1.0, 3.0, 5.0]).to(DEVICE), label_smoothing=0.05),
        'spleen': nn.CrossEntropyLoss(weight=torch.tensor([1.0, 3.0, 5.0]).to(DEVICE), label_smoothing=0.05),
    }

    for param in raw_model.backbone.parameters():
        param.requires_grad = False

    optimizer = get_optimizer_with_llrd(model, base_lr=LEARNING_RATE)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    scaler = GradScaler('cuda', enabled=(DEVICE.type == 'cuda')) # DEVICE가 object이므로 .type 추가 권장

    start_epoch = 0
    checkpoint_data = None
    history = {
        "train_loss": [],
        "val_loss": [],
        "auc_avg_loss": [],  # 이전에 만든 AUC도 기록
        "auc_details": [],
    }

# --- [체크포인트 로드] ---
    if os.path.exists(MONAI_MODEL_SAVE_PATH_CONTINUE) and MONAI_MODEL_SAVE_PATH_CONTINUE != '':
        print(f"\n🔍 체크포인트 발견: {MONAI_MODEL_SAVE_PATH_CONTINUE}")
        checkpoint_data = torch.load(MONAI_MODEL_SAVE_PATH_CONTINUE, map_location=DEVICE)
        
        # 모델 가중치 복원
        state_dict = checkpoint_data['model']
        new_state_dict = OrderedDict()
        for k, v in state_dict.items():
            name = k[7:] if k.startswith('module.') else k
            new_state_dict[name] = v
        raw_model.load_state_dict(new_state_dict)

        start_epoch = checkpoint_data['epoch'] + 1
        
        # 5에폭 이상에서 중단된 경우를 위해 가중치를 불러온 후 requires_grad 상태를 먼저 맞춰줌
        if start_epoch >= 5:
            for param in model.parameters(): param.requires_grad = True
            # 옵티마이저 재구성 (로드 전 구조를 맞추기 위함)
            optimizer = get_optimizer_with_llrd(model, base_lr=LEARNING_RATE, layer_decay=LAYER_DECAY)
            scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

        optimizer.load_state_dict(checkpoint_data['optimizer'])
        scheduler.load_state_dict(checkpoint_data['scheduler'])
        scaler.load_state_dict(checkpoint_data['scaler'])
        history = checkpoint_data['history']
        print(f"✅ {start_epoch} 에포크부터 재개합니다.")


    for epoch in range(NUM_EPOCHS):
        if epoch < start_epoch: continue
            
        # --- [스테이지 전환: 5에폭에서 동결 해제] ---
        if epoch == 5:
            print(f"\n🔓 [Epoch {epoch}] 전체 동결 해제 및 옵티마이저 재설정")
            for param in model.parameters():
                param.requires_grad = True
            
            # 파라미터 그룹이 바뀌었으므로 옵티마이저만 새로 생성 (스케줄러는 유지 가능하나 새로 생성 권장)
            optimizer = get_optimizer_with_llrd(model, base_lr=LEARNING_RATE, layer_decay=LAYER_DECAY)
            scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
        

        model.train()
        train_epoch_loss = 0
        accumulation_steps = 8
        optimizer.zero_grad() # 루프 시작 전 초기화

        train_loop = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch}/{NUM_EPOCHS} [Train]", leave=False)
        for i, batch in train_loop:
            inputs = batch["image"].to(DEVICE)

            with autocast(device_type=DEVICE.type, enabled=(DEVICE.type == 'cuda')):
                outputs = model(inputs)

                # Loss 계산 (초기화 중요)
                loss = 0
                for k in CLASS_NAME_LIST:
                    target = batch[k].to(DEVICE)
                    if hasattr(target, "as_tensor"):
                        target = target.as_tensor() # 메타데이터 미리 제거

                    if target.dim() > 1:
                        target = torch.argmax(target, dim=1)
                    target = target.long() # 이제 target은 확실한 순수 텐서

                    # 예측값도 안전하게 처리
                    pred_k = outputs[k].as_tensor() if hasattr(outputs[k], "as_tensor") else outputs[k]
                    loss_func = criterion_dict[k]

                    if k == 'any_injury':
                        # 문지기인 any_injury는 손실값 자체에도 배율을 주어 학습을 리드하게 함
                        loss += loss_func(pred_k, target) * 2
                    else:
                        loss += loss_func(pred_k, target)
                        
                loss = loss / accumulation_steps
            
            scaler.scale(loss).backward()
            # 4. accumulation_steps마다 또는 마지막 배치일 때 가중치 업데이트
            if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad() # 기울기 초기화

            train_epoch_loss += (loss.item() * accumulation_steps) # 원래 loss 값으로 복원해서 기록
            train_loop.set_postfix(loss=(loss.item() * accumulation_steps))

        avg_train_loss = train_epoch_loss / len(train_loader)


        # 에포크 종료 후 성능 출력
        auc_results, avg_val_loss = evaluate(model, val_loader, epoch, criterion_dict) # 실무에선 val_loader 사용 권장
        mean_auc = sum(auc_results.values()) / len(auc_results)
        
        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(avg_val_loss)
        history["auc_avg_loss"].append(mean_auc)
        history["auc_details"].append(auc_results)

        # CosineAnnealingLR 사용 시 (에포크 끝날 때마다 호출)
        # scheduler.step()

        # ReduceLROnPlateau 사용시
        scheduler.step(avg_val_loss)

        print(f"\n>>> Epoch {epoch} Summary")
        print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
        print(f"Mean AUC: {mean_auc:.4f}")
        for organ, val in auc_results.items():
            print(f" - {organ:15s}: {val:.4f}")
        print("-" * 50)

        # 5. 모델 가중치 저장
        raw_model = model.module if hasattr(model, 'module') else model
        save_dict = {
            'epoch': epoch,
            'model': raw_model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(), # 나중에 이어할 때 필수
            'scaler': scaler.state_dict(),
            'history': history
        }
        current_save_path = model_save_path.replace(".pth", f"_ep{epoch}.pth")
        torch.save(save_dict, f"{current_save_path}")
        print(f"✅ 모델 가중치 저장 완료: {current_save_path}")

        # 6. 학습 히스토리 저장 (Pickle)
        history_save_path = model_save_path.replace(".pth", ".pkl")
        with open(history_save_path, 'wb') as file:
            pickle.dump(history, file)
        print(f"✅ 학습 히스토리 저장 완료: {history_save_path}")

    return history


def show_history(history):
    plt.figure(figsize=(15, 6))

    # 1. Loss 그래프 (Training vs Validation)
    plt.subplot(1, 3, 1)
    plt.plot(history["train_loss"], label="Train Loss", marker='o')
    plt.plot(history["val_loss"], label="Val Loss", marker='o')
    plt.title("Training & Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.legend()

    # 2. Mean AUC 그래프
    # 키 이름을 val_auc_mean으로 수정했습니다.
    plt.subplot(1, 3, 2)
    plt.plot(history["auc_avg_loss"], label="Mean Val AUC", color='orange', marker='s')
    plt.title("Mean Validation AUC")
    plt.xlabel("Epoch")
    plt.ylabel("AUC")
    plt.grid(True)
    plt.legend()

    # 장기별로 리스트를 추출하여 그래프 그리기
    plt.subplot(1, 3, 3) # 1행 3열 중 3번째 (에러 해결 지점)
    for organ in CLASS_NAME_LIST:
        # 각 장기별 데이터를 추출하여 루프 안에서 그립니다.
        organ_auc_history = [epoch_data[organ] for epoch_data in history["auc_details"]]
        plt.plot(organ_auc_history, label=f"{organ}")

    plt.title("Validation AUC by Organ")
    plt.xlabel("Epoch")
    plt.ylabel("AUC")
    plt.ylim(0.4, 1.05) # AUC가 1일 수도 있으므로 1.05 정도로 설정
    plt.grid(True, linestyle='--')
    # 범례가 많을 수 있으므로 그래프 옆으로 뺍니다.
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small')

    plt.tight_layout()
    plt.show()


In [ ]:
train_files_subset = train_files[:]

train_results = []
for item in train_files_subset:
    train_results.append(process_one_item(item))

# 에포크 에러 방지를 위해 None 제거 (주소록 업데이트)
train_files_preprocessed = [r for r in train_results if r is not None]

val_files_subset = val_files[:]

val_results = []
for item in val_files_subset:
    val_results.append(process_one_item(item))

# 에포크 에러 방지를 위해 None 제거 (주소록 업데이트)
val_files_preprocessed = [r for r in val_results if r is not None]

# 학습
monai_train_loader_pipeline = monai_train_pipeline()
monai_val_loader_pipeline = monai_val_pipeline()

print("=" * 25,"Train","=" * 25)
history = train(train_files_preprocessed, val_files_preprocessed,
                monai_train_loader_pipeline, monai_val_loader_pipeline,
                MONAI_MODEL_SAVE_PATH)

show_history(history)